# 13b: Measurement and site shift, and the UCI Heart main result

Design decisions
- Measurement shift (NHANES): one XGBoost trained like the frozen heart model (frozen settings, seed 42, Youden threshold on 5-fold out-of-fold scores) on the BRFSS 2015 training portion, with the 16 predictors NHANES shares. Education and income are coarsened as in notebook 03; diabetes is coded yes or no, because NHANES asks about prediabetes as its own question (about 17% say yes, against about 2% in BRFSS). Prior from 1,000 training patients, certificates on calibration split 1.
- Targets: NHANES with self-reported values, then the same people with measured BMI and measured high blood pressure in place of the self-reported items. Both use the same 500 explained patients, the same records for checking claims and the same re-certification draws, so the difference between them is the measurement alone. Re-certification at 500 to 5,000 records (NHANES has 6,287 adults), 20 draws, at least 500 records left to check claims. The no-shift reference is the BRFSS 2015 test split.
- Site shift (UCI): Cleveland split 50/35/15 (stratified) into training, calibration and test rows. XGBoost settings (100 trees, 32 bins, depth 2 to 4, minimum child weight 1 or 5) are chosen on the Cleveland training rows only, by 5-fold out-of-fold AUROC. Targets Hungary, Switzerland and VA: every record is explained and also used to check claims. Re-certification is not run, because each target site has fewer than 300 records.
- UCI Heart main result (E1, the plan's leave-one-site-out design): for each site, a model built the same way on the other three sites is evaluated on the held-out site.
- UCI claims are evaluated at the frozen heart targets (alpha_0 = 0.05, alpha_1 = 0.70) and at the most lenient grid point (alpha_0 = 0.10, beta = 0.2, so alpha_1 = 0.80). The policy test is not run: there is no second calibration split at this size.
- UCI models are exported with inputs converted to float32, as XGBoost does (ST depression has one decimal); the measurement model, on integer-coded data, is exported like the frozen models.
- Transport checks, domain-classifier diagnostics and label-shift estimates are those of notebook 13. Model metrics (AUROC, AUPRC, Brier score, ECE) are reported for every target.
- Every finished piece is saved on Drive, so an interrupted run resumes. Expected run time on Colab CPU: about 30 to 45 minutes.

In [ ]:
# === SESSION BOOTSTRAP (run first) ===
import subprocess, sys, warnings


def quiet_warnings():
    # imodels switches on every DeprecationWarning; Colab's message code then warns on every output line
    warnings.filterwarnings('ignore', category=DeprecationWarning, module='jupyter_client')
    warnings.filterwarnings('ignore', message=r'datetime\.datetime\.utcnow')


quiet_warnings()
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'z3-solver'], check=True)
from google.colab import drive
drive.mount('/content/drive')
import os, shutil
DRIVE_ROOT = '/content/drive/MyDrive/SCAX_Research'
REPO_DIR = f'{DRIVE_ROOT}/scax-research'
for name in ('.gitconfig', '.git-credentials'):
    if os.path.exists(f'{DRIVE_ROOT}/{name}'):
        shutil.copy(f'{DRIVE_ROOT}/{name}', os.path.expanduser(f'~/{name}'))
os.chdir(REPO_DIR)
os.environ['SCAX_DRIVE_ROOT'] = DRIVE_ROOT
sys.path.insert(0, f'{REPO_DIR}/src')
pull = subprocess.run(['git', 'pull', 'origin', 'main'], capture_output=True, text=True)
print(pull.stdout.strip() or pull.stderr.strip())

In [ ]:
import json, re, time, zlib
import numpy as np
import pandas as pd
from scax import paths
from scax.utils import seed_everything, sha256_file, write_json
from scax.data import public, splits
from scax.data import brfss_raw as br
from scax.data import nhanes as nh
from scax.data import uci
from scax.certify.clopper_pearson import cp_lower, cp_upper
from scax.experiments.main_results import FrozenTask
from scax.metrics.model_metrics import model_metrics
from scax.models.export import Ensemble
from scax.experiments import shift as sh
from scax.experiments import model_ablations as ma
from scax.experiments.model_ablations import fit_variant
import inspect
if not hasattr(sh, 'choose_settings') or not hasattr(ma, 'exported_vs_native_detail') \
        or 'input_cast' not in inspect.signature(fit_variant).parameters:
    raise RuntimeError('The stage 11 source is not loaded. Run the stage 11 batch, then Runtime > Disconnect and delete runtime, then run again.')

SEED = 42
seed_everything(SEED)
TASK = 'brfss2015_heart'
KIND = {0: 'negative outcome', 1: 'enrichment'}
REC05 = json.load(open(f'{paths.RESULTS}/05_frozen.json'))
FT = FrozenTask(TASK, REPO_DIR, DRIVE_ROOT, REC05['frozen_config_sha256'][TASK])
REC02 = json.load(open(f'{paths.RESULTS}/02_brfss_rebuild.json'))
REC03 = json.load(open(f'{paths.RESULTS}/03_nhanes_uci.json'))
F15 = paths.DERIVED / 'brfss' / 'brfss_2015.parquet'
NH_FILE = paths.DERIVED / 'nhanes' / 'nhanes_2015_2018.parquet'
UCI_FILE = paths.DERIVED / 'uci' / 'uci_heart_sites.parquet'
assert sha256_file(F15) == REC02['parquet_sha256']['2015'], 'brfss_2015.parquet differs from the file notebook 02 wrote'
assert sha256_file(NH_FILE) == REC03['nhanes']['parquet_sha256'], 'the NHANES table differs from the one notebook 03 wrote'
assert sha256_file(UCI_FILE) == REC03['uci']['parquet_sha256'], 'the UCI table differs from the one notebook 03 wrote'
B15 = pd.read_parquet(F15)
code = np.load(paths.DERIVED / 'splits' / f'{TASK}.npy')
assert splits.summary(code)['sha256'] == FT.cfg['split_sha256'], 'split differs from the frozen one'
FULL = list(FT.ens.feature_names)
X_pub, y_pub = public.LOADERS[TASK](paths.RAW)
assert len(B15) == len(code) and np.array_equal(B15[FULL].to_numpy(float), X_pub[FULL].to_numpy(float)) \
    and np.array_equal(B15[br.TARGET].to_numpy(int), y_pub), 'the rebuilt 2015 table must equal the public table row by row'
del X_pub, y_pub

PARAMS = FT.cfg['model']['params']
CAND = FT.cfg['candidates']
ALPHA = {0: FT.targets['alpha_0'], 1: FT.targets['alpha_1']}
LENIENT = {0: 0.10, 1: 0.80}
DELTA1 = FT.targets['delta_1']
N_PRIOR, N_REF, N_NH = 1000, 500, 500
SIZES, DRAWS, MIN_EVAL = [500, 1000, 2000, 5000], 20, 500
UCI_GRID = [{'n_estimators': 100, 'max_bin': 32, 'max_depth': d, 'min_child_weight': w} for d in (2, 3, 4) for w in (1, 5)]
SHIFT_DIR = paths.DERIVED / 'shift13b'
for d in ('models', 'results'):
    (SHIFT_DIR / d).mkdir(parents=True, exist_ok=True)
print('frozen configuration', FT.cfg_sha[:16], '| settings', PARAMS, '| alpha', ALPHA, '| delta_1', DELTA1)

In [ ]:
MODELS, NATIVE, RESULTS, DEV = {}, {}, [], []
LOG = lambda m: print(m, flush=True)


def seed_of(*parts):
    return zlib.crc32('|'.join(map(str, parts)).encode()) % (2 ** 31)


def slug(text):
    return re.sub(r'[^A-Za-z0-9]+', '_', text).strip('_')


def build_model(name, X_tr, y_tr, names, X_prior, X_cal, y_cal, params, input_cast=None):
    """One fixed model (the fit is deterministic), checked against its saved copy; then its prior and certificates."""
    t0 = time.time()
    model, ens, pred = fit_variant('xgboost', X_tr, y_tr, params, SEED, names, input_cast=input_cast)
    f = SHIFT_DIR / 'models' / f'{name}__model.json'
    if f.exists():
        assert Ensemble.load(f).hash() == ens.hash(), f'{name}: refitting does not reproduce the saved model'
    else:
        ens.save(f)
    agree = ma.exported_vs_native_detail(model, 'xgboost', pred, X_cal)
    assert agree['beyond_rounding'] == 0, f'{name}: exported and native models disagree beyond float32 rounding: {agree}'
    if agree['within_rounding']:
        print(f"{name}: {agree['within_rounding']} calibration rows lie within float32 rounding of the threshold "
              f"(largest distance {agree['max_gap']:.1e}, bound {agree['rounding_bound']:.1e}); the exported exact sum decides them")
    sm = sh.ShiftModel(name, pred, CAND['M'], CAND['order_seed'], CAND['node_budget'], FT.lam, ALPHA, DELTA1,
                       cache_path=SHIFT_DIR / 'models' / f'{name}__candidates.pkl')
    sm.set_prior(X_prior, log=LOG)
    sm.set_calibration(X_cal, y_cal)
    NATIVE[name] = model
    MODELS[name] = {'model_sha256': ens.hash(), 'predictors': list(names), 'settings': dict(params), 'input_cast': ens.input_cast,
                    'training_records': int(len(y_tr)), 'training_prevalence': float(np.mean(y_tr)),
                    'calibration_records': int(len(y_cal)), 'prior_patients': int(len(X_prior)),
                    'prior_boxes': len(sm.counts), 'thresholds': int(sum(len(t) for t in pred.grid.thresholds)),
                    'tau': float(ens.tau), 'exported_vs_native': agree}
    print(f"{name}: {len(y_tr):,} training records, {MODELS[name]['thresholds']} thresholds, {len(y_cal):,} calibration "
          f"records, hash {ens.hash()[:12]}, {time.time() - t0:.0f}s", flush=True)
    return sm


def run_target(sm, setting, target, role, X_t, y_t, n_explain=None, split_seed=None, sizes=(), alpha=None,
               targets_set='frozen', min_records=50):
    """Transport (and re-certification when sizes are given) for one target; saved on Drive when finished."""
    res_file = SHIFT_DIR / 'results' / f'{sm.name}__{slug(target)}__{targets_set}.json'
    if res_file.exists():
        r = json.load(open(res_file))
        assert r['model_sha256'] == sm.model_hash, f'{res_file.name} was computed with a different model'
        RESULTS.append(r); print(f'{sm.name} | {target} | {targets_set}: loaded from Drive', flush=True)
        return r
    t0 = time.time()
    assert sm.model_hash == MODELS[sm.name]['model_sha256']
    sm.check_hash()
    s = seed_of(sm.name, target) if split_seed is None else split_seed
    if n_explain is None or n_explain >= len(y_t):
        exp = rest = np.arange(len(y_t))            # every record explained, claims checked on the same records
    else:
        exp = np.sort(np.random.default_rng(s).choice(len(y_t), n_explain, replace=False))
        rest = np.setdiff1d(np.arange(len(y_t)), exp)
    te = sh.TargetEval(sm, X_t[exp], y_t[exp], X_t[rest], y_t[rest], log=LOG, alpha=alpha)
    met = te.transport()
    recert = te.recertify(list(sizes), DRAWS, s, MIN_EVAL) if sizes else []
    i_s, i_t, p_s, p_t, dom = sh.domain_scores(sm.X_cal, X_t[rest], s)
    ids = np.unique(te.rel_transport[te.rel_transport >= 0])
    boxes = sh.box_diagnostics(te, ids, sm.C_cal[i_s], te.C_rest[i_t], p_s, p_t, min_records=min_records)
    mm = model_metrics(NATIVE[sm.name].predict_proba(np.asarray(X_t, dtype=np.float64))[:, 1], y_t,
                       sm.pred.predict(X_t).astype(int))
    info = {**sh.model_summary(sm.pred, X_t, y_t), 'auprc': mm['auprc'], 'brier': mm['brier'],
            'ece_15_equal_mass': mm['ece_15_equal_mass'], 'calibration_slope': mm['calibration_slope'],
            'explained': int(len(exp)), 'checking_records': int(len(rest)), 'source_prevalence': float(sm.y_cal.mean()),
            'bbse_prevalence': sh.bbse_prevalence(sm.yhat_cal, sm.y_cal, sm.pred.predict(X_t[rest])),
            'domain_auroc': dom, 'candidate_boxes': len(te.keys), 'seed': int(s)}
    r = {'setting': setting, 'model': sm.name, 'target': target, 'role': role, 'targets_set': targets_set,
         'alpha': {str(k): v for k, v in te.alpha.items()}, 'model_sha256': sm.model_hash, 'info': info,
         'transport': [{'kind': k, **met[k]} for k in (0, 1)], 'recert': recert,
         'boxes': boxes.round(6).to_dict('records'), 'seconds': time.time() - t0}
    write_json(res_file, r)
    RESULTS.append(r)
    print(f"{sm.name} | {target} | {targets_set}: AUROC {info['auroc']:.3f}, prevalence {info['prevalence']:.1%}, "
          f"domain AUROC {dom:.3f}; source certificates explain {met[0]['coverage']:.1%} / {met[1]['coverage']:.1%} "
          f"(negative / enrichment), errors {met[0]['error_rate']:.1%} / {met[1]['error_rate']:.1%}, claims refuted "
          f"{met[0]['refuted_share']:.1%} / {met[1]['refuted_share']:.1%}; {time.time() - t0:.0f}s", flush=True)
    return r

## Measurement shift: a BRFSS 2015 model on NHANES, self-reported and then measured

In [ ]:
def brfss16(df):
    c = nh.brfss_common(df)
    c['Diabetes'] = (c.Diabetes == 2).astype(float)       # prediabetes coded as no in both surveys
    return c.to_numpy(float)


def nhanes16(t, measured):
    X = t[nh.N_PREDICTORS].astype(float).copy()
    X['Diabetes'] = (X.Diabetes == 2).astype(float)
    if measured:
        X['BMI'] = t.BMI_measured.astype(float)
        X['HighBP'] = t.HighBP_measured.astype(float)
    return X.to_numpy(float)


XB, y15 = brfss16(B15), B15[br.TARGET].to_numpy(int)
tr, cal1, test = (splits.indices(code, s) for s in ('train', 'cal1', 'test'))
prior = np.sort(np.random.default_rng(seed_of('prior', 'measurement')).choice(tr, N_PRIOR, replace=False))
sm = build_model('measurement_brfss2015_common16', XB[tr], y15[tr], nh.N_PREDICTORS, XB[prior], XB[cal1], y15[cal1], PARAMS)
run_target(sm, 'measurement', 'BRFSS 2015 test split', 'reference', XB[test], y15[test], N_REF, sizes=SIZES)
NHT = pd.read_parquet(NH_FILE)
yN = NHT[nh.TARGET].to_numpy(int)
NH_SEED = seed_of('nhanes', 'explained patients')
for label, measured in (('NHANES, self-reported', False), ('NHANES, measured BMI and blood pressure', True)):
    run_target(sm, 'measurement', label, 'target', nhanes16(NHT, measured), yN, N_NH, split_seed=NH_SEED, sizes=SIZES)
del sm

## Site shift (Cleveland to the other sites) and the UCI main result (each site held out)

In [ ]:
U = pd.read_parquet(UCI_FILE)
XU, yU, SITE = U[uci.PREDICTORS].to_numpy(float), U[uci.TARGET].to_numpy(int), U.site.to_numpy()


def uci_model(name, source):
    idx = np.flatnonzero(source)
    part = sh.split_small(yU[idx], seed_of('uci split', name))
    tr_, cal_, te_ = (idx[part == k] for k in (0, 1, 2))
    best, table = sh.choose_settings(XU[tr_], yU[tr_], UCI_GRID, SEED)
    DEV.append(table.assign(model=name))
    print(f'{name}: settings chosen on {len(tr_)} training rows by out-of-fold AUROC: depth {best["max_depth"]}, '
          f'minimum child weight {best["min_child_weight"]} (AUROC {table.oof_auroc.max():.3f})')
    # ST depression has one decimal, so the export converts inputs to float32 exactly as XGBoost does
    return build_model(name, XU[tr_], yU[tr_], uci.PREDICTORS, XU[tr_], XU[cal_], yU[cal_], best, input_cast='float32'), te_


SETS = (('frozen', None), ('lenient', LENIENT))
sm, te_ = uci_model('site_cleveland', SITE == 'cleveland')
for name, alpha in SETS:
    run_target(sm, 'site', 'Cleveland test split', 'reference', XU[te_], yU[te_], alpha=alpha, targets_set=name, min_records=20)
    for s in ('hungarian', 'switzerland', 'va'):
        m = SITE == s
        run_target(sm, 'site', uci.SITE_NAMES[s], 'target', XU[m], yU[m], alpha=alpha, targets_set=name, min_records=20)
for held in uci.SITES:
    sm, te_ = uci_model(f'uci_without_{held}', SITE != held)
    for name, alpha in SETS:
        run_target(sm, 'uci_main', f'{uci.SITE_NAMES[held]} held out: source test split', 'reference', XU[te_], yU[te_],
                   alpha=alpha, targets_set=name, min_records=20)
        m = SITE == held
        run_target(sm, 'uci_main', f'{uci.SITE_NAMES[held]} held out', 'target', XU[m], yU[m], alpha=alpha,
                   targets_set=name, min_records=20)
del sm

## Results

In [ ]:
ID = ('setting', 'model', 'target', 'role', 'targets_set')
TGT = pd.DataFrame([{**{k: r[k] for k in ID}, **r['info'], 'seconds': r['seconds']} for r in RESULTS])
TRN = pd.DataFrame([{**{k: r[k] for k in ID}, 'alpha_claim': r['alpha'][str(row['kind'])], **row}
                    for r in RESULTS for row in r['transport']])
DRW = pd.DataFrame([{**{k: r[k] for k in ID}, **row} for r in RESULTS for row in r['recert']])
REC = pd.concat([sh.summarise_recert(r['recert']).assign(**{k: r[k] for k in ID}) for r in RESULTS if r['recert']],
                ignore_index=True)
BOX = pd.DataFrame([{**{k: r[k] for k in ID}, **row} for r in RESULTS for row in r['boxes']])
TRN = TRN.merge(TGT[['model', 'target', 'targets_set', 'explained']], on=['model', 'target', 'targets_set'])
TRN['coverage_lo'] = [float(cp_lower(a, n, 0.05)) for a, n in zip(TRN.accepted, TRN.explained)]
TRN['coverage_hi'] = [float(cp_upper(a, n, 0.05)) for a, n in zip(TRN.accepted, TRN.explained)]
TRN['kind_name'] = TRN.kind.map(KIND); REC['kind_name'] = REC.kind.map(KIND)
DEVT = pd.concat(DEV, ignore_index=True)
pd.set_option('display.width', 250)
COLS = ['targets_set', 'target', 'explained', 'predicted', 'accepted', 'coverage', 'errors', 'error_rate', 'error_upper_95',
        'refuted_share', 'above_target_share', 'certificate_refuted_share', 'mean_size']
print(TGT[['setting', 'target', 'targets_set', 'records', 'prevalence', 'auroc', 'auprc', 'brier', 'ece_15_equal_mass',
           'sensitivity', 'specificity', 'bbse_prevalence', 'domain_auroc']].round(3).to_string(index=False))
for setting in ('measurement', 'site', 'uci_main'):
    for k in (0, 1):
        print(f'\n{setting}: {KIND[k]} claims with source certificates')
        print(TRN[(TRN.setting == setting) & (TRN.kind == k)][COLS].round(4).to_string(index=False))
for k in (0, 1):
    print(f'\nmeasurement shift, re-certification with labelled target records, {KIND[k]} claims (mean of {DRAWS} draws)')
    piv = REC[(REC.setting == 'measurement') & (REC.kind == k)].pivot_table(index='target', columns='m', values='coverage', sort=False)
    print(piv.round(3).to_string())
bad = REC[REC.draws_with_refuted_claims > 0]
print('draws with a claim refuted on target records:', 'none' if bad.empty else bad[['target', 'kind', 'm', 'draws_with_refuted_claims']].to_string(index=False))
print('\nUCI settings chosen on training rows (out-of-fold AUROC):')
print(DEVT.pivot_table(index='model', columns=['max_depth', 'min_child_weight'], values='oof_auroc').round(3).to_string())

## Figure

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.ticker import FixedFormatter, FixedLocator, NullLocator, PercentFormatter
plt.rcParams.update({'font.family': 'DejaVu Sans', 'font.size': 8, 'axes.spines.top': False, 'axes.spines.right': False})
STYLE = {'BRFSS 2015 test split': ('0.6', 's', 'No shift (BRFSS 2015 test split)'),
         'NHANES, self-reported': ('#0072B2', 'o', 'NHANES, self-reported values'),
         'NHANES, measured BMI and blood pressure': ('#D55E00', 'o', 'NHANES, measured BMI and blood pressure')}
fig, axes = plt.subplots(1, 2, figsize=(7.4, 3.3))
for ax, k in zip(axes, (0, 1)):
    for t, (col, mk, lab) in STYLE.items():
        s = REC[(REC.setting == 'measurement') & (REC.target == t) & (REC.kind == k)].sort_values('m')
        if t != 'BRFSS 2015 test split':
            ax.fill_between(s.m, s.coverage_q025, s.coverage_q975, color=col, alpha=0.13, lw=0)
        ax.plot(s.m, s.coverage, color=col, lw=1.3 if mk == 'o' else 1.6, marker=mk, ms=2.8, label=lab)
        tr = TRN[(TRN.setting == 'measurement') & (TRN.target == t) & (TRN.kind == k)].coverage
        if len(tr):
            ax.axhline(float(tr.iloc[0]), color=col, lw=0.9, ls=(0, (4, 2)))
    ax.set_xscale('log')
    ax.xaxis.set_major_locator(FixedLocator(SIZES))
    ax.xaxis.set_major_formatter(FixedFormatter(['500', '1k', '2k', '5k'][:len(SIZES)]))
    ax.xaxis.set_minor_locator(NullLocator())
    ax.set_ylim(bottom=0)
    ax.yaxis.set_major_formatter(PercentFormatter(1.0, decimals=0))
    ax.grid(color='0.92', lw=0.6)
    ax.set_title(f'{KIND[k].capitalize()} claims', fontsize=8.5)
    ax.set_xlabel('Labelled target records used for the certificates')
axes[0].set_ylabel('Share of explained patients')
h, l = axes[0].get_legend_handles_labels()
fig.legend(h, l, loc='upper center', ncol=3, frameon=False, fontsize=7, bbox_to_anchor=(0.5, 1.03))
fig.text(0.5, -0.02, f'Solid: certificates from labelled target records (mean of {DRAWS} draws; band: middle 95% of draws). '
         'Dashed: source certificates, no target labels.', ha='center', fontsize=6.5, color='0.3')
fig.tight_layout(rect=(0, 0.02, 1, 0.92))
for ext in ('png', 'pdf'):
    fig.savefig(paths.FIGURES / f'fig_shift_measurement.{ext}', dpi=600, bbox_inches='tight')
plt.show()

In [ ]:
TRN.to_csv(f'{paths.RESULTS}/13b_shift.csv', index=False)
TGT.to_csv(f'{paths.RESULTS}/13b_shift_targets.csv', index=False)
REC.to_csv(f'{paths.RESULTS}/13b_shift_recertification.csv', index=False)
DRW.to_csv(f'{paths.RESULTS}/13b_shift_recertification_draws.csv', index=False)
BOX.to_csv(f'{paths.RESULTS}/13b_shift_boxes.csv', index=False, float_format='%.6g')
DEVT.to_csv(f'{paths.RESULTS}/13b_uci_settings.csv', index=False)
write_json(f'{paths.RESULTS}/13b_shift.json', {
    'seed': SEED, 'frozen_config_sha256': FT.cfg_sha, 'brfss_2015_parquet_sha256': REC02['parquet_sha256']['2015'],
    'nhanes_parquet_sha256': REC03['nhanes']['parquet_sha256'], 'uci_parquet_sha256': REC03['uci']['parquet_sha256'],
    'targets': {'frozen': ALPHA, 'lenient_uci': LENIENT, 'delta_1': DELTA1},
    'sizes': {'prior_patients': N_PRIOR, 'explained_reference': N_REF, 'explained_nhanes': N_NH,
              'recertification_sizes': SIZES, 'draws': DRAWS, 'min_checking_records': MIN_EVAL},
    'predictors_measurement': nh.N_PREDICTORS, 'predictors_uci': uci.PREDICTORS, 'uci_settings_grid': UCI_GRID,
    'models': MODELS, 'seconds_total': float(TGT.seconds.sum())})
print('written: results/13b_shift.csv, 13b_shift_targets.csv, 13b_shift_recertification.csv, '
      '13b_shift_recertification_draws.csv, 13b_shift_boxes.csv, 13b_uci_settings.csv, 13b_shift.json; '
      'figures/fig_shift_measurement')

In [ ]:
# === STANDARD COMMIT CELL ===
import os, subprocess
os.chdir(REPO_DIR)
r = subprocess.run(["python", "tools/commit_cell.py",
  "13b: measurement shift (BRFSS model on NHANES, self-reported then measured) and site shift (UCI Cleveland to other sites), UCI main result leave-one-site-out, figure"],
  capture_output=True, text=True)
print(r.stdout); print(r.stderr)